<h2 align="center">Naive Bayes Model<h2>

<p>Trying to train basic language identification Naive Bayes model in intra-word code-switching<p>

## Import needed libraries 

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

## Load The Data

In [4]:
DataPath = '../../Data Collection and Annotation/Data.csv'
data = pd.read_csv(DataPath)
display(data.head(10))
data.shape

,#,Sentence#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,1,derasa,AR,AR,derasa,6
1,2,1,?,OTHER,OTHER,?,1
2,3,1,ya,AR,AR,ya,2
3,4,1,addy,NE.AR,NE.AR,addy,4
4,5,1,howa,AR,AR,howa,4
5,6,1,2enta,AR,AR,2enta,5
6,7,1,betro7a,AR,AR,betro7a,7
7,8,1,elgam3a,AR,AR,elgam3a,7
8,9,1,?🙂,OTHER,OTHER,?🙂,2
9,10,2,aywa,AR,AR,aywa,4


(30321, 7)

In [3]:
# extract the (token,segment,tag,SegmentLen) from the data
agg_func = lambda s: [(w, p, t, l) for w, p, t, l in zip(s["Token"].values.tolist(),s["Segment"].values.tolist(),s["Tag-Mixed"].values.tolist(),s["SegmentLen"].values.tolist())]

# grouped the data with its sentence to separate the sentence by a newline
grouped = data.groupby("Sentence#").apply(agg_func)
sentences = [s for s in grouped]

# split the sentences into train and test sets by 800:229 ratio
train, test = train_test_split(sentences, test_size=0.222546161, random_state=2018)

## Data Statistics

In [21]:
tag = data['Tag'].value_counts()
mixedTag = data['Tag-Mixed'].value_counts()
tag, mixedTag

(AR       19007
 EN        6110
 OTHER     3794
 MIXED      777
 NE.AR      317
 NE.EN      250
 AMBIG       55
 LANG3       11
 Name: Tag, dtype: int64,
 AR                19007
 EN                 6110
 OTHER              3794
 AR,EN               715
 NE.AR               317
 NE.EN               250
 AMBIG                55
 EN,AR                29
 AR,EN,AR             24
 LANG3                11
 AR,OTHER,EN           8
 AR,OTHER,EN,AR        1
 Name: Tag-Mixed, dtype: int64)

In [32]:
# Split data into raw features and labels

X_train = []
for s in train:
    for w in s:
        X_train.append(w[0])
y_train = []
for s in train:
    for w in s:
        y_train.append(w[2])
X_test =  []
for s in test:
    for w in s:
        X_test.append(w[0])
y_test = []
for s in test:
    for w in s:
        y_test.append(w[2])

## Train The Model 

In [33]:
# Make Machine Learning Pipeline with TfidfVectorizer and MultinomialNB
tfidf_vect = TfidfVectorizer(analyzer='char', ngram_range=(1,3))
model = MultinomialNB()
text_clf = Pipeline([('tfidf', tfidf_vect),
                    ('clf', model),
                    ])

In [36]:
# Train model with pipeline classifier
text_clf.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(1, 3))),
                ('clf', MultinomialNB())])

In [37]:
# Make predictions on test data with trained classifier
predictions = text_clf.predict(X_test)

In [38]:
# Measure accuracy
accuracy_score(y_test,predictions)

0.8914841143188742

In [41]:
# Print the classification report for all test data including the F-score
print(classification_report(y_test, predictions,digits=6))

              precision    recall  f1-score   support

       AMBIG   0.000000  0.000000  0.000000        15
          AR   0.905014  0.981868  0.941876      4357
       AR,EN   0.000000  0.000000  0.000000       174
    AR,EN,AR   0.000000  0.000000  0.000000         8
 AR,OTHER,EN   0.000000  0.000000  0.000000         3
          EN   0.789993  0.867647  0.827001      1292
       EN,AR   0.000000  0.000000  0.000000         4
       LANG3   0.000000  0.000000  0.000000         2
       NE.AR   0.000000  0.000000  0.000000       100
       NE.EN   0.000000  0.000000  0.000000        66
       OTHER   0.998661  0.855505  0.921557       872

    accuracy                       0.891484      6893
   macro avg   0.244879  0.245911  0.244585      6893
weighted avg   0.846460  0.891484  0.866943      6893



In [43]:
# Print the classification report for mixed test data only including the F-score
tags = ["AR","EN","NE.AR","NE.EN","OTHER","AMBIG","LANG3"]
acts,preds = [],[]
for act, pred in zip(y_test,predictions):
    if(not(act in tags) or not(pred in tags)):
        acts.append(act)
        preds.append(pred)
print(classification_report(acts,preds,digits=6))

              precision    recall  f1-score   support

          AR   0.000000  0.000000  0.000000       0.0
       AR,EN   0.000000  0.000000  0.000000     174.0
    AR,EN,AR   0.000000  0.000000  0.000000       8.0
 AR,OTHER,EN   0.000000  0.000000  0.000000       3.0
          EN   0.000000  0.000000  0.000000       0.0
       EN,AR   0.000000  0.000000  0.000000       4.0

    accuracy                       0.000000     189.0
   macro avg   0.000000  0.000000  0.000000     189.0
weighted avg   0.000000  0.000000  0.000000     189.0



In [44]:
def test_languages(X_test, true_values, predictions):

    true_values = np.array(true_values)
    
    for i, sentence in enumerate(X_test):
        prediction = predictions[i]
        true_value = true_values[i]
        print("Prediction: " + str(prediction))
        print("Actual Language: " + str(true_value))
        print("Input Sentence: ")
        print(sentence)
        print('\n')

# Visualize Performance
test_languages(X_test, y_test, predictions)

Prediction: AR
Actual Language: AR
Input Sentence: 
أنا


Prediction: AR
Actual Language: AR
Input Sentence: 
متفق


Prediction: AR
Actual Language: AR
Input Sentence: 
معاك


Prediction: AR
Actual Language: AR
Input Sentence: 
إن


Prediction: AR
Actual Language: AR
Input Sentence: 
الجو


Prediction: AR
Actual Language: AR
Input Sentence: 
برد


Prediction: AR
Actual Language: AR
Input Sentence: 
لكن


Prediction: AR
Actual Language: AR
Input Sentence: 
مش


Prediction: AR
Actual Language: AR
Input Sentence: 
معنى


Prediction: AR
Actual Language: AR
Input Sentence: 
كده


Prediction: AR
Actual Language: AR
Input Sentence: 
انك


Prediction: AR
Actual Language: AR
Input Sentence: 
تبوظ


Prediction: AR
Actual Language: AR,EN
Input Sentence: 
اللاب


Prediction: AR
Actual Language: EN
Input Sentence: 
توب


Prediction: AR
Actual Language: AR
Input Sentence: 
بتاعك


Prediction: AR
Actual Language: AR
Input Sentence: 
لو


Prediction: AR
Actual Language: AR
Input Sentence: 
مضطر


Pred

Actual Language: EN
Input Sentence: 
leave


Prediction: EN
Actual Language: EN
Input Sentence: 
ur


Prediction: EN
Actual Language: EN
Input Sentence: 
fucking


Prediction: EN
Actual Language: EN
Input Sentence: 
opinion


Prediction: EN
Actual Language: EN
Input Sentence: 
to


Prediction: EN
Actual Language: EN
Input Sentence: 
urself


Prediction: AR
Actual Language: AR
Input Sentence: 
begad


Prediction: EN
Actual Language: EN
Input Sentence: 
it


Prediction: EN
Actual Language: EN
Input Sentence: 
doesn’t


Prediction: AR
Actual Language: EN
Input Sentence: 
make


Prediction: AR
Actual Language: AR
Input Sentence: 
any


Prediction: EN
Actual Language: EN
Input Sentence: 
sense


Prediction: EN
Actual Language: EN
Input Sentence: 
and


Prediction: EN
Actual Language: EN
Input Sentence: 
you


Prediction: EN
Actual Language: EN
Input Sentence: 
won’t


Prediction: EN
Actual Language: EN
Input Sentence: 
get


Prediction: AR
Actual Language: AR
Input Sentence: 
any


Predicti

Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
‏


Prediction: AR
Actual Language: AR
Input Sentence: 
3al


Prediction: AR
Actual Language: AR
Input Sentence: 
7atab


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
‏


Prediction: EN
Actual Language: EN
Input Sentence: 
eatery


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
‏


Prediction: EN
Actual Language: NE.AR
Input Sentence: 
andiamo


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
_______________


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
-


Prediction: AR
Actual Language: AR
Input Sentence: 
أقوي


Prediction: AR
Actual Language: AR
Input Sentence: 
مطاعم


Prediction: AR
Actual Language: AR
Input Sentence: 
للمطبخ


Prediction: AR
Actual Language: AR
Input Sentence: 
العالمي


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
:


Prediction: EN
Actual Language: EN
Input Sentence: 
eatery


Prediction: EN
Actual Language: NE.AR
Input Sentence: 
sedra


Pred



Prediction: AR
Actual Language: AR
Input Sentence: 
msln


Prediction: AR
Actual Language: OTHER
Input Sentence: 
5


Prediction: EN
Actual Language: EN
Input Sentence: 
is


Prediction: EN
Actual Language: EN
Input Sentence: 
represented


Prediction: AR
Actual Language: EN
Input Sentence: 
as


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
0


Prediction: EN
Actual Language: EN
Input Sentence: 
and


Prediction: EN
Actual Language: EN
Input Sentence: 
so


Prediction: EN
Actual Language: EN
Input Sentence: 
on


Prediction: AR
Actual Language: AR
Input Sentence: 
fa


Prediction: AR
Actual Language: AR
Input Sentence: 
enta


Prediction: AR
Actual Language: AR
Input Sentence: 
3yz


Prediction: AR
Actual Language: AR
Input Sentence: 
t3mel


Prediction: EN
Actual Language: EN
Input Sentence: 
subtraction


Prediction: AR
Actual Language: AR
Input Sentence: 
fa


Prediction: EN
Actual Language: EN
Input Sentence: 
assuming


Prediction: AR
Actual Language: AR
Input Sente

Actual Language: OTHER
Input Sentence: 
5


Prediction: AR
Actual Language: AR
Input Sentence: 
yaani


Prediction: AR
Actual Language: AR
Input Sentence: 
kda


Prediction: AR
Actual Language: AR
Input Sentence: 
kda


Prediction: AR
Actual Language: AR
Input Sentence: 
ramdan


Prediction: AR
Actual Language: AR
Input Sentence: 
alla


Prediction: AR
Actual Language: AR
Input Sentence: 
allah


Prediction: AR
Actual Language: AR
Input Sentence: 
حد


Prediction: AR
Actual Language: AR
Input Sentence: 
هنا


Prediction: AR
Actual Language: AR
Input Sentence: 
عنده


Prediction: AR
Actual Language: AR
Input Sentence: 
امتحان


Prediction: AR
Actual Language: AR
Input Sentence: 
الماني


Prediction: AR
Actual Language: AR
Input Sentence: 
يوم


Prediction: AR
Actual Language: AR
Input Sentence: 
الاحد


Prediction: AR
Actual Language: AR
Input Sentence: 
او


Prediction: AR
Actual Language: AR
Input Sentence: 
الاربعاء


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
؟


Pred



Prediction: AR
Actual Language: AR,EN
Input Sentence: 
elmedia


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
?


Prediction: AR
Actual Language: AR
Input Sentence: 
hwa


Prediction: AR
Actual Language: AR
Input Sentence: 
l


Prediction: AR
Actual Language: AR
Input Sentence: 
so2al


Prediction: AR
Actual Language: AR
Input Sentence: 
3yz


Prediction: EN
Actual Language: AR,OTHER,EN
Input Sentence: 
y-accept


Prediction: AR
Actual Language: AR
Input Sentence: 
eh


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
?


Prediction: EN
Actual Language: EN
Input Sentence: 
no


Prediction: EN
Actual Language: EN
Input Sentence: 
need


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
♥️♥️


Prediction: EN
Actual Language: EN
Input Sentence: 
answer


Prediction: EN
Actual Language: EN
Input Sentence: 
to


Prediction: AR
Actual Language: EN
Input Sentence: 
how


Prediction: EN
Actual Language: EN
Input Sentence: 
do


Prediction: EN
Actual Language: EN

... (8 outputs left out)

Prediction: EN
Actual Language: EN
Input Sentence: 
the


Prediction: AR
Actual Language: EN
Input Sentence: 
same


Prediction: AR
Actual Language: AR
Input Sentence: 
hwa


Prediction: AR
Actual Language: AR
Input Sentence: 
m7dsh


Prediction: AR
Actual Language: AR
Input Sentence: 
etkalem


Prediction: AR
Actual Language: AR
Input Sentence: 
3n


Prediction: AR
Actual Language: AR
Input Sentence: 
l


Prediction: EN
Actual Language: EN
Input Sentence: 
project


Prediction: EN
Actual Language: EN
Input Sentence: 
so


Prediction: AR
Actual Language: AR
Input Sentence: 
kol


Prediction: AR
Actual Language: AR
Input Sentence: 
wa7ed


Prediction: AR
Actual Language: AR
Input Sentence: 
by3mel


Prediction: EN
Actual Language: AR,EN
Input Sentence: 
blimagination


Prediction: AR
Actual Language: AR
Input Sentence: 
bta3o


Prediction: AR
Actual Language: AR
Input Sentence: 
w


Prediction: AR
Actual Language: AR
Input Sentence: 
keda


Prediction: AR
Actual Language: AR
Input Sente

Actual Language: AR
Input Sentence: 
mmkn


Prediction: AR
Actual Language: AR
Input Sentence: 
kol


Prediction: EN
Actual Language: EN
Input Sentence: 
tweet


Prediction: AR
Actual Language: AR
Input Sentence: 
bt2hr


Prediction: AR
Actual Language: AR
Input Sentence: 
aktar


Prediction: AR
Actual Language: AR
Input Sentence: 
mn


Prediction: AR
Actual Language: AR
Input Sentence: 
eli


Prediction: AR
Actual Language: AR
Input Sentence: 
ablaha


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
😂💔💔


Prediction: AR
Actual Language: AR
Input Sentence: 
hwa


Prediction: AR
Actual Language: AR
Input Sentence: 
ana


Prediction: AR
Actual Language: AR
Input Sentence: 
ezay


Prediction: AR
Actual Language: AR
Input Sentence: 
ab2a


Prediction: AR
Actual Language: AR
Input Sentence: 
wa5ed


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
1


Prediction: AR
Actual Language: EN
Input Sentence: 
lab


Prediction: EN
Actual Language: AMBIG
Input Sentence: 
hascal




ba2aly


Prediction: AR
Actual Language: OTHER
Input Sentence: 
2


Prediction: EN
Actual Language: EN
Input Sentence: 
weeks


Prediction: EN
Actual Language: EN
Input Sentence: 
or


Prediction: EN
Actual Language: EN
Input Sentence: 
more


Prediction: AR
Actual Language: AR
Input Sentence: 
mashoftahash


Prediction: AR
Actual Language: AR
Input Sentence: 
w


Prediction: AR
Actual Language: AR
Input Sentence: 
awel


Prediction: AR
Actual Language: AR
Input Sentence: 
mahadantaha


Prediction: EN
Actual Language: AR
Input Sentence: 
haset


Prediction: AR
Actual Language: AR
Input Sentence: 
any


Prediction: AR
Actual Language: AR
Input Sentence: 
ba2et


Prediction: AR
Actual Language: AR
Input Sentence: 
kowayesa


Prediction: AR
Actual Language: AR
Input Sentence: 
w


Prediction: AR
Actual Language: EN
Input Sentence: 
an


Prediction: AR
Actual Language: AR
Input Sentence: 
kolo


Prediction: AR
Actual Language: AR
Input Sentence: 
hyb2a


Prediction: AR
Actual Language: AR


Actual Language: OTHER
Input Sentence: 
-


Prediction: AR
Actual Language: OTHER
Input Sentence: 
3


Prediction: EN
Actual Language: EN
Input Sentence: 
not


Prediction: EN
Actual Language: EN
Input Sentence: 
sure


Prediction: EN
Actual Language: EN
Input Sentence: 
tho


Prediction: AR
Actual Language: AR
Input Sentence: 
kolo


Prediction: AR
Actual Language: AR
Input Sentence: 
da5el


Prediction: AR
Actual Language: AR
Input Sentence: 
هو


Prediction: AR
Actual Language: AR
Input Sentence: 
انا


Prediction: AR
Actual Language: AR
Input Sentence: 
ماشي


Prediction: AR
Actual Language: AR
Input Sentence: 
بالسبت


Prediction: AR
Actual Language: AR
Input Sentence: 
عادي


Prediction: AR
Actual Language: AR
Input Sentence: 
والله


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
😂


Prediction: AR
Actual Language: EN
Input Sentence: 
wtf


Prediction: AR
Actual Language: AR
Input Sentence: 
hwa


Prediction: OTHER
Actual Language: OTHER
Input Sentence: 
30


Predicti

In [45]:
# test the model with some sentences
predictions = text_clf.predict(["ana","raeh","ellab","now"])
predictions

array(['AR', 'AR', 'AR', 'EN'], dtype='<U14')

In [46]:
predictions = text_clf.predict(["أنا","رايح","اللاب","ناو"])
predictions

array(['AR', 'AR', 'AR', 'AR'], dtype='<U14')